In [9]:
import requests
import pandas as pd
import io
import matplotlib.pyplot as plt
from astropy.io import fits

# 1. Search for the latest 50 LSST alerts classified as SN Ia
# Note: In 2026, we use the LSST-specific subdomain
API_URL = "https://api.fink-portal.org/api/v1/latests"

r = requests.post(
    API_URL,
    json={
        'class': 'SN Ia', 
        'n': '50', 
        'output-format': 'json',}
        )

sn_list = pd.read_json(io.BytesIO(r.content))

# 2. Grab a specific cutout (Science image) for the first SN in the list
first_sn = sn_list.iloc[0]

cutout_r = requests.post(
    'https://api.lsst.fink-portal.org/api/v1/cutouts',
    json={
        'objectId': first_sn['i:objectId'],
        'kind': 'Science',  # Options: 'Science', 'Template', 'Difference'
        'output-format': 'FITS'
    }
)

# 3. Open the image directly in memory
with fits.open(io.BytesIO(cutout_r.content)) as hdul:
    data = hdul[0].data
    plt.imshow(data, cmap='gray', origin='lower')
    plt.title(f"SN Ia Candidate: {first_sn['i:objectId']}")
    plt.show()

IndexError: single positional indexer is out-of-bounds

In [4]:
import requests
import pandas as pd
import io
from astropy.io import fits

# 1. Get 50 SN Ia Object IDs
r = requests.post(
    'https://api.fink-portal.org/api/v1/latests',
    json={'class': 'SN Ia', 'n': 50, }
)
objects = pd.read_json(io.BytesIO(r.content))

# 2. Download their Science Cutouts as FITS
for _, row in objects.iterrows():
    img_data = requests.post(
        'https://api.fink-portal.org/api/v1/cutouts',
        json={'objectId': row['i:objectId'], 'kind': 'Science', 'output-format': 'FITS'}
    )
    # Save to your data/raw/fink_alerts/ folder
    with open(f"C:\\Users\\riley\\OneDrive\\Documents\\University\\Honours\\Honours_Project\\data\\raw\\fink_alerts\\{row['i:objectId']}.fits", "wb") as f:
        f.write(img_data.content)

In [2]:
import requests
import pandas as pd
import io
import os
from astropy.io import fits

# 1. Configuration - Set your survey (ZTF or LSST)
# For ZTF: https://api.fink-portal.org/api/v1
# For LSST: https://api.lsst.fink-portal.org/api/v1
BASE_URL = 'https://api.fink-portal.org/api/v1'
SAVE_DIR = "data/raw/fink_alerts"

# Create the directory if it doesn't exist
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Searching for SN Ia candidates...")

# 2. Get 50 SN Ia Object IDs using the /latests endpoint
# We use /latests because we are searching by CLASS, not specific ID
r = requests.post(
    f'{BASE_URL}/latests',
    json={
        'class': 'Early SN Ia candidate', 
        'n': '50' 
    }
)

if r.status_code != 200:
    print(f"Error fetching list: {r.text}")
else:
    objects = pd.read_json(io.BytesIO(r.content))
    print(f"Found {len(objects)} objects. Starting download...")

    # 3. Download their Science Cutouts as FITS
    for _, row in objects.iterrows():
        obj_id = row['i:objectId']
        
        # Download the cutout
        img_data = requests.post(
            f'{BASE_URL}/cutouts',
            json={
                'objectId': obj_id, 
                'kind': 'Science', 
                'output-format': 'FITS'
            }
        )
        
        if img_data.status_code == 200:
            file_path = os.path.join(SAVE_DIR, f"{obj_id}.fits")
            with open(file_path, "wb") as f:
                f.write(img_data.content)
            print(f" Successfully saved: {obj_id}.fits")
        else:
            print(f" Failed to download cutout for {obj_id}: {img_data.status_code}")

print("\nTask Complete!")

Searching for SN Ia candidates...
Error fetching list: <html>
<head><title>504 Gateway Time-out</title></head>
<body>
<center><h1>504 Gateway Time-out</h1></center>
<hr><center>nginx/1.20.1</center>
</body>
</html>


Task Complete!


In [ ]:
import requests
import pandas as pd
import io
import os

# Configuration
BASE_URL = 'https://api.fink-portal.org/api/v1'
SAVE_DIR = "data/raw/fink_alerts"
# The 'kinds' available in Fink: Science, Template, Difference
KINDS = ['Science', 'Template', 'Difference']

for kind in KINDS:
    os.makedirs(os.path.join(SAVE_DIR, kind), exist_ok=True)

# 1. Using the list you just generated (assuming it's in 'objects' dataframe)
# If you need to re-run the search:
r = requests.post(f'{BASE_URL}/latests', json={'class': 'Early SN Ia candidate', 'n': '50'})
objects = pd.read_json(io.BytesIO(r.content))

print(f"Starting Triple-Cutout Download for {len(objects)} objects...")

for _, row in objects.iterrows():
    obj_id = row['i:objectId']
    
    for kind in KINDS:
        img_data = requests.post(
            f'{BASE_URL}/cutouts',
            json={'objectId': obj_id, 'kind': kind, 'output-format': 'FITS'}
        )
        
        if img_data.status_code == 200:
            file_name = f"{obj_id}_{kind}.fits"
            file_path = os.path.join(SAVE_DIR, kind, file_name)
            with open(file_path, "wb") as f:
                f.write(img_data.content)
        else:
            print(f" [!] Missing {kind} for {obj_id}")

print("Done! Check your 'data/raw/fink_alerts' folder.")

   d:DR3Name  d:Plx  d:anomaly_score  d:blazar_stats_m0  d:blazar_stats_m1  \
0        NaN    NaN        -0.099223                 -1                 -1   
1        NaN    NaN        -0.112508                 -1                 -1   
2        NaN    NaN        -0.120990                 -1                 -1   
3        NaN    NaN        -0.100555                 -1                 -1   
4        NaN    NaN        -0.099417                 -1                 -1   

   d:blazar_stats_m2 d:cdsxmatch  d:delta_time  d:e_Plx  d:from_upper  ...  \
0                 -1     Unknown      1.000521      NaN         False  ...   
1                 -1     Unknown      1.971968      NaN         False  ...   
2                 -1     Unknown      0.104942      NaN         False  ...   
3                 -1     Unknown      0.000937      NaN         False  ...   
4                 -1     Unknown      0.960833      NaN         False  ...   

   i:ssnamenr  i:tooflag     i:xpos     i:ypos  d:tracklet  \
